In [1]:
from wppkg import generate_default_debugpy_config

In [2]:
generate_default_debugpy_config()

In [1]:
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [15]:
adata = sc.read_h5ad('/data/home/zhangyaojie/Lung_stack/data/preneo_merge_NG_GM.h5ad')

In [3]:
print(adata)
print(adata.obs)
print(adata.obs['response'].unique())

AnnData object with n_obs × n_vars = 72586 × 23647
    obs: 'sample_id', 'response', 'source'
                                               sample_id response  source
BD_immune05_545070-GM                        BD_immune05     NMPR      GM
BD_immune05_804090-GM                        BD_immune05     NMPR      GM
BD_immune05_458030-GM                        BD_immune05     NMPR      GM
BD_immune05_775827-GM                        BD_immune05     NMPR      GM
BD_immune05_42202-GM                         BD_immune05     NMPR      GM
...                                                  ...      ...     ...
XGY_P_P10_N_TTCACGCATCTTCACAACAGCAGA-NG_pre  XGY_P_P10_N     NMPR  NG_pre
XGY_P_P10_N_TTCACGCATCTTCACACCTCCTGA-NG_pre  XGY_P_P10_N     NMPR  NG_pre
XGY_P_P10_N_TTCACGCATGAAGAGAAATGTTGC-NG_pre  XGY_P_P10_N     NMPR  NG_pre
XGY_P_P10_N_TTCACGCATGGAACAACATCAAGT-NG_pre  XGY_P_P10_N     NMPR  NG_pre
XGY_P_P10_N_TTCACGCATGGTGGTAAGCACCTC-NG_pre  XGY_P_P10_N     NMPR  NG_pre

[72586 rows x 3 c

In [7]:
sample_response = (
    adata.obs[["sample_id", "response"]]
    .drop_duplicates()
    .sort_values("sample_id")
)
sample_response

,sample_id,response
BD_P_P01_N_219824-NG_pre,BD_P_P01_N,NMPR
BD_immune05_545070-GM,BD_immune05,NMPR
BD_immune08_219824-GM,BD_immune08,NMPR
XGY_P_P05_P_AAACATCGCTGTAGCCCAACCACA-NG_pre,XGY_P_P05_P,pCR
XGY_P_P06_M_AAACATCGAACCGAGAGTACGCAA-NG_pre,XGY_P_P06_M,MPR
XGY_P_P07_N_AAACATCGAACAACCAAAACATCG-NG_pre,XGY_P_P07_N,NMPR
XGY_P_P08_N_AAACATCGAAGACGGACAACCACA-NG_pre,XGY_P_P08_N,NMPR
XGY_P_P09_P_AAACATCGACAGATTCACAGATTC-NG_pre,XGY_P_P09_P,pCR
XGY_P_P10_N_AAACATCGACACAGAAATCATTCC-NG_pre,XGY_P_P10_N,NMPR


In [4]:
adata = sc.read_h5ad('/data/home/zhangyaojie/data/HJJ_NSCLC/all_raw.h5ad')

In [5]:
metadata = pd.read_csv('/data/home/zhangyaojie/data/HJJ_NSCLC/hjj_metadata.csv')

adata_samples = set(adata.obs["sample"].unique())
metadata_samples = set(metadata["Sample"].unique())

In [6]:
missing_in_meta = sorted(adata_samples - metadata_samples)
print(missing_in_meta)
extra_in_meta = sorted(metadata_samples - adata_samples)
print(extra_in_meta)
common_samples = sorted(adata_samples & metadata_samples)
print("\n两边共有 sample 数量:", len(common_samples))

['P17']
[]

两边共有 sample 数量: 178


In [10]:
adata

AnnData object with n_obs × n_vars = 2666754 × 58336
    obs: 'n_genes', 'total_counts', 'doublet_scores', 'predicted_doublets', 'sample_lineage', 'sample', 'batch', 'pct_counts_mt'
    var: 'gene_id', 'gene_type'

In [7]:
adata.obs["sample"] = adata.obs["sample"].astype(str)
metadata["Sample"] = metadata["Sample"].astype(str)
#将metadata中的Sample列名称修改为sample
metadata.rename(columns={"Sample": "sample"}, inplace=True)

In [8]:
adata.obs = adata.obs.join(
    metadata.set_index("sample"),
    on="sample"
)

In [9]:
print(adata)
print(adata.obs)

AnnData object with n_obs × n_vars = 2666754 × 58336
    obs: 'n_genes', 'total_counts', 'doublet_scores', 'predicted_doublets', 'sample_lineage', 'sample', 'batch', 'pct_counts_mt', 'ORR', 'RECIST', 'Patient', 'ResTumor', 'Response', 'Type', 'Type2', 'Match', 'Time', 'Time2', 'Regimen', 'Regimen2', 'Regimen3', 'Regimen4', 'Group', 'Group2', 'Group3', 'Group4', 'Group5', 'NewIndex'
    var: 'gene_id', 'gene_type'
                                   n_genes  total_counts  doublet_scores  \
barcodes                                                                   
P187T_AACACACAGACCAGGTCATTAGCATCC     1107        3020.0        0.010878   
P187T_AACACACAGACCGTACTCCGACTGAGA     2707       10997.0        0.036411   
P187T_AACACACAGACCTCGACTAGCACATGC     1632        6590.0        0.222506   
P187T_AACACACAGACCTCGACTCTGTTCGGT      895        2146.0        0.033499   
P187T_AACACACAGACCTGCTACTGCTATCGC      888        1560.0        0.024109   
...                                    ...         

In [10]:
adata.obs['Regimen'].unique()

array(['IAT', 'ICT', 'CT', nan], dtype=object)

In [11]:
adata = adata[adata.obs['Time'] == "Baseline"]
adata = adata[adata.obs['Response'] != "NE"]

adata_ICT = adata[adata.obs['Regimen'] == "ICT"]
adata_IAT = adata[adata.obs['Regimen'] == "IAT"]

In [12]:
print(adata_ICT)
print(adata_ICT.obs)
print(adata_ICT.X[:20, :20].toarray())
print(adata_IAT)
print(adata_IAT.obs)
print(adata_IAT.X[:20, :20].toarray())

View of AnnData object with n_obs × n_vars = 287511 × 58336
    obs: 'n_genes', 'total_counts', 'doublet_scores', 'predicted_doublets', 'sample_lineage', 'sample', 'batch', 'pct_counts_mt', 'ORR', 'RECIST', 'Patient', 'ResTumor', 'Response', 'Type', 'Type2', 'Match', 'Time', 'Time2', 'Regimen', 'Regimen2', 'Regimen3', 'Regimen4', 'Group', 'Group2', 'Group3', 'Group4', 'Group5', 'NewIndex'
    var: 'gene_id', 'gene_type'
                             n_genes  total_counts  doublet_scores  \
barcodes                                                             
P9_AAACATCGAACAACCACACCTTAC      570        1052.0        0.009510   
P9_AAACATCGAACAACCATGAAGAGA      330         589.0        0.026140   
P9_AAACATCGAATCCGTCAATCCGTC      840        1456.0        0.140549   
P9_AAACATCGAATCCGTCATCCTGTA      760        2154.0        0.122655   
P9_AAACATCGACAAGCTACGCATACA      476         976.0        0.058629   
...                              ...           ...             ...   
P3_TTCACGCATGGTG

In [13]:
adata_ICT.obs = pd.DataFrame(
    {
        "sample_id": adata_ICT.obs["sample"].astype(str).values,
        "response": adata_ICT.obs["Response"].astype(str).values,
        "source": "hjj",
    },
    index=adata_ICT.obs_names,
)
adata_ICT

AnnData object with n_obs × n_vars = 287511 × 58336
    obs: 'sample_id', 'response', 'source'
    var: 'gene_id', 'gene_type'

In [28]:
print(adata.var_names)
print(adata_ICT.var_names)
print(adata_ICT.var['gene_type'])

Index(['A1BG', 'A1BG-AS1', 'A1CF', 'A2M', 'A2M-AS1', 'A2ML1', 'A4GALT', 'AAAS',
       'AACS', 'AADAC',
       ...
       'ZW10', 'ZWILCH', 'ZWINT', 'ZXDA', 'ZXDB', 'ZXDC', 'ZYG11A', 'ZYG11B',
       'ZYX', 'ZZEF1'],
      dtype='object', length=23647)
Index(['5S_rRNA_1', '5S_rRNA_2', '5S_rRNA_3', '5_8S_rRNA_1', '5_8S_rRNA_2',
       '5_8S_rRNA_3', '5_8S_rRNA_4', '5_8S_rRNA_5', '7SK_1', '7SK_2',
       ...
       'ZXDB', 'ZXDC', 'ZYG11A', 'ZYG11AP1', 'ZYG11B', 'ZYX', 'ZYXP1', 'ZZEF1',
       'hsa-mir-1253', 'hsa-mir-3607'],
      dtype='object', name='gene_name', length=58336)
gene_name
5S_rRNA_1                         rRNA
5S_rRNA_2                         rRNA
5S_rRNA_3                         rRNA
5_8S_rRNA_1                       rRNA
5_8S_rRNA_2                       rRNA
                         ...          
ZYX                     protein_coding
ZYXP1           unprocessed_pseudogene
ZZEF1                   protein_coding
hsa-mir-1253                   lincRNA
hsa-mir-3607    

In [16]:
adata_merged = ad.concat(
    [adata, adata_ICT],
    axis=0,
    join="outer",
    merge="first",
    label=None,
    fill_value=0,
)
adata_merged

AnnData object with n_obs × n_vars = 360097 × 58631
    obs: 'sample_id', 'response', 'source'
    var: 'gene_id', 'gene_type'

In [17]:
adata_merged.var = adata_merged.var.iloc[:, 0:0].copy()

In [18]:
adata_merged

AnnData object with n_obs × n_vars = 360097 × 58631
    obs: 'sample_id', 'response', 'source'

In [ ]:
print(adata_merged.obs)
print(adata_merged.obs['sample_id'].unique())
for i in adata_merged.obs['sample_id'].unique():
    print(i, adata_merged.obs[adata_merged.obs['sample_id'] == i].shape[0])
    print(adata_merged.obs[adata_merged.obs['sample_id'] == i]['response'].unique())

                               sample_id response source
BD_immune05_545070-GM        BD_immune05     NMPR     GM
BD_immune05_804090-GM        BD_immune05     NMPR     GM
BD_immune05_458030-GM        BD_immune05     NMPR     GM
BD_immune05_775827-GM        BD_immune05     NMPR     GM
BD_immune05_42202-GM         BD_immune05     NMPR     GM
...                                  ...      ...    ...
P3_TTCACGCATGGTGGTAGAACAGGC           P3     NMPR    hjj
P3_TTCACGCATGGTGGTAGCCACATA           P3     NMPR    hjj
P3_TTCACGCATTCACGCAACAGATTC           P3     NMPR    hjj
P3_TTCACGCATTCACGCAACGTATCA           P3     NMPR    hjj
P3_TTCACGCATTCACGCAGAGCTGAA           P3     NMPR    hjj

[360097 rows x 3 columns]
['BD_immune05' 'BD_immune08' 'BD_P_P01_N' 'XGY_P_P05_P' 'XGY_P_P06_M'
 'XGY_P_P07_N' 'XGY_P_P08_N' 'XGY_P_P09_P' 'XGY_P_P10_N' 'P9' 'P29' 'P11'
 'P33' 'P10' 'P19' 'P5' 'P7' 'P15' 'P21' 'P6' 'P30' 'P31' 'P1' 'P2' 'P27'
 'P12' 'P18' 'P22' 'P8' 'P3']
BD_immune05 3113
['NMPR']
BD_immune08 582

In [27]:
adata_merged.write_h5ad('/data/home/zhangyaojie/Lung_stack/data/preneo_merge_NG_GM_hjj.h5ad')

In [28]:
adata_IAT.obs = pd.DataFrame(
    {
        "sample_id": adata_IAT.obs["sample"].astype(str).values,
        "response": adata_IAT.obs["Response"].astype(str).values,
        "source": "hjj",
    },
    index=adata_IAT.obs_names,
)
adata_IAT

AnnData object with n_obs × n_vars = 377593 × 58336
    obs: 'sample_id', 'response', 'source'
    var: 'gene_id', 'gene_type'

In [40]:
adata_IAT.var = adata_IAT.var.iloc[:, 0:0].copy()

In [ ]:
adata_IAT.obs['sample_id'].unique()

array(['P46', 'P57', 'P42', 'P44', 'P34', 'P54', 'P41', 'P47', 'P52',
       'P55', 'P45', 'P50', 'P16', 'P39', 'P38', 'P4', 'P25', 'P20',
       'P56', 'P37', 'P48', 'P49', 'P53', 'P40', 'P14', 'P43', 'P35'],
      dtype=object)

In [42]:
adata_IAT.write_h5ad('/data/home/zhangyaojie/Lung_stack/data/preneo_IAT.h5ad')

In [4]:
adata_ICT = sc.read_h5ad('/data/home/zhangyaojie/Lung_stack/data/preneo_merge_NG_GM_hjj.h5ad')
# adata_IAT = sc.read_h5ad('/data/home/zhangyaojie/Lung_stack/data/preneo_IAT.h5ad')

In [ ]:
adata_ICT.obs['sample_id'] == ""

In [5]:
print(adata_ICT.obs.groupby('sample_id').size())
# print(adata_IAT.obs.groupby('sample_id').size())

sample_id
BD_P_P01_N      5607
BD_immune05     3113
BD_immune08     5825
P1             18704
P2             12855
P3             11162
P5             22571
P6             13472
P7             18481
P8             22983
P9              4180
P10             5793
P11            17999
P12            19394
P15             7164
P18            18556
P19            14181
P21            16211
P22            13077
P27            13644
P29             9360
P30             9951
P31             5621
P33            12152
XGY_P_P05_P     5836
XGY_P_P06_M    14843
XGY_P_P07_N    10312
XGY_P_P08_N    12081
XGY_P_P09_P     2339
XGY_P_P10_N    12630
dtype: int64


/tmp/ipykernel_67517/4041949537.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(adata_ICT.obs.groupby('sample_id').size())


In [14]:
adata_ICT.obs[adata_ICT.obs['sample_id'] == "XGY_P_P05_P"]['response'].unique()

['pCR']
Categories (3, object): ['MPR', 'NMPR', 'pCR']